# Práctica 2: Análisis Exploratorio de Datos (EDA)
## Actividad 6: Tratamiento de Desbalance - Telco Customer Churn

En este notebook se audita el desbalance de la variable objetivo de abandono (`Churn`), caracterizada por un desbalance moderado o alto asociado a la pérdida de clientes, aplicando las metodologías correspondientes[cite: 1, 3].

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

# 1. Cargar el dataset de Telco Churn
df_telco = pd.read_csv('../data/Telco-Customer-Churn.csv')

# Limpieza y preparación específica para Telco
df_telco['TotalCharges'] = pd.to_numeric(df_telco['TotalCharges'], errors='coerce')
df_telco.fillna(0, inplace=True)

# Mapear la variable objetivo a valores binarios (1 para Yes/Churn, 0 para No)
if 'Churn' in df_telco.columns:
    df_telco['Target'] = df_telco['Churn'].apply(lambda x: 1 if str(x).strip().lower() in ['yes', '1', 'true'] else 0)

# Seleccionamos algunas variables numéricas clave de ejemplo (tenure, MonthlyCharges, TotalCharges)
cols_num_telco = ['tenure', 'MonthlyCharges', 'TotalCharges']

X = df_telco[cols_num_telco]
y = df_telco['Target']

### 1. Análisis de Frecuencias y Cálculo del IR (Telco Churn)
Determinamos el volumen de clientes que abandonan el servicio frente a los que permanecen y calculamos la razón de desbalance ($IR$).

In [ ]:
conteo_telco = y.value_counts()
porcentajes_telco = y.value_counts(normalize=True) * 100

print("--- Conteo de observaciones (Churn) ---")
print(conteo_telco)
print("\n--- Porcentaje por clase ---")
print(porcentajes_telco)

mayoritaria_t = conteo_telco.idxmax()
minoritaria_t = conteo_telco.idxmin()
IR_telco = conteo_telco[mayoritaria_t] / conteo_telco[minoritaria_t]

print(f"\nRazón de desbalance (IR): {IR_telco:.2f}")
if IR_telco > 1.5:
    print("-> El dataset Telco Churn está desbalanceado (IR > 1.5)[cite: 3].")
else:
    print("-> El dataset está balanceado.")

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 2. Comparativa de Estrategias de Tratamiento (Telco)
Evaluamos los efectos del sobremuestreo, submuestreo, ponderación y SMOTE sobre las métricas de predicción de abandono de clientes[cite: 1, 3].

In [ ]:
estrategias_telco = {
    "1. Sin balanceo (Línea base)": (X_train_t, y_train_t, None),
    "2. Ponderación de clases": (X_train_t, y_train_t, "balanced"),
    "3. Sobremuestreo aleatorio": *RandomOverSampler(random_state=42).fit_resample(X_train_t, y_train_t), "none",
    "4. Submuestreo aleatorio": *RandomUnderSampler(random_state=42).fit_resample(X_train_t, y_train_t), "none",
    "5. SMOTE": *SMOTE(random_state=42).fit_resample(X_train_t, y_train_t), "none"
}

for nombre, datos in estrategias_telco.items():
    if len(datos) == 3:
        X_tr, y_tr, cw = datos
        modelo = LogisticRegression(class_weight=cw, random_state=42, max_iter=1000)
    else:
        X_tr, y_tr, _ = datos
        modelo = LogisticRegression(random_state=42, max_iter=1000)
    
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_test_t)
    
    print(f"================ {nombre} ================")
    print(classification_report(y_test_t, y_pred, zero_division=0))